# 01 — LASH and Fair Benchmark

This notebook reproduces the **dataset-local HPO protocol used for the revised manuscript**.

Key safeguards:

- the four datasets are optimized independently;
- HPO uses only Train and `Val_tune`;
- each stochastic family uses the same two HPO seeds (`42`, `142`);
- the dimension-adaptive budget is **3–5 finite COMPLETE trials per repeat**;
- pruned/failed trials are never eligible for selection;
- finalists are confirmed on the complete validation-tuning segment;
- final stochastic refits use seeds `42`, `142`, and `242`;
- no prior result directory is imported and no Test target enters selection.

Generated artifacts are written under `outputs/` and are intentionally excluded from Git version control.


## 1. Repository paths and deterministic runtime

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, platform
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Must be set before torch is imported.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "42")
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(key, "1")

# %pip install -r ../requirements.txt
# Install the PyTorch CPU/CUDA build appropriate for your machine separately.


## 2. Exact paper protocol

In [ ]:
import pandas as pd
import torch
import psutil, cpuinfo
from IPython.display import display

import lash_revision_core as core
from lash_revision_core import ExperimentConfig
from lash_hardware_optimized import apply_hardware_patch, probe_gpu_boosters
from lash_per_dataset_hpo import (
    PROTOCOL_NAME,
    PerDatasetHPOPolicy,
    protocol_manifest,
    search_space_manifest,
    run_per_dataset_hpo_benchmark,
)

apply_hardware_patch()
booster_probe = probe_gpu_boosters()

physical_cores = psutil.cpu_count(logical=False) or 1
logical_threads = psutil.cpu_count(logical=True) or physical_cores
ram_gb = psutil.virtual_memory().total / 1024**3
cpu_name = cpuinfo.get_cpu_info().get("brand_raw") or platform.processor() or "Unknown CPU"

# The reported experiments used Ryzen 7 7800X3D + RTX 5070 Ti.
# CPU execution remains possible, but will be substantially slower.
TREE_HORIZON_JOBS = min(8, max(1, physical_cores))
torch.set_num_threads(min(8, max(1, logical_threads)))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

policy = PerDatasetHPOPolicy(computational_budget_hours=48.0)

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    dataset_keys=("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM"),
    run_profile="paper",
    weather_mode="historical_only",
    hpo_seeds=policy.hpo_seeds,                 # (42, 142)
    final_refit_seeds=policy.lash_final_seeds,  # (42, 142, 242)
    primary_hpo_repeats=2,
    external_hpo_repeats=2,
    hpo_trials_per_dimension=1,
    hpo_min_trials=3,
    hpo_max_trials=5,
    max_epochs=policy.max_epochs,
    early_stopping_patience=policy.patience,
    tree_horizon_jobs=TREE_HORIZON_JOBS,
    tree_threads_per_model=1,
    require_cuda=False,
    use_amp=True,
    save_models=True,
    resume=True,
)

hardware = {
    "os": platform.platform(),
    "python": platform.python_version(),
    "cpu": cpu_name,
    "physical_cpu_cores": physical_cores,
    "logical_cpu_threads": logical_threads,
    "ram_gb": round(ram_gb, 2),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only",
    "gpu_vram_gb": (
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        if torch.cuda.is_available() else 0.0
    ),
    "cuda_build": str(torch.version.cuda),
    "tree_horizon_jobs": TREE_HORIZON_JOBS,
    "tree_threads_per_model": 1,
    "automatic_mixed_precision": bool(config.amp_enabled),
    **booster_probe,
}
display(pd.DataFrame([hardware]))
display(protocol_manifest(policy, config.dataset_keys))


## 3. Freeze the run contract before model fitting

In [ ]:
EXPECTED_CSVS = (
    "Cluster_1_Harmonized.csv",
    "Cluster_2_Harmonized.csv",
    "BDG_Edu_Harmonized.csv",
    "BDG_Dorm_Harmonized.csv",
)

input_rows = []
for filename in EXPECTED_CSVS:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    required = ["Date", "Holi", "Temp", "Humi", "WS", "Consumption"]
    if list(frame.columns) != required:
        raise ValueError(f"{filename}: expected {required}, got {list(frame.columns)}")
    input_rows.append({
        "file": filename,
        "rows": len(frame),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })

CONTRACT_PATH = OUTPUT_DIR / "run_contract_20260823_per_dataset_hpo.json"
contract = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "protocol": PROTOCOL_NAME,
    "selection_scope": "independent dataset-local train + validation-tune HPO",
    "prior_result_reuse": False,
    "identical_hpo_rule_across_datasets": True,
    "hpo_seeds": list(policy.hpo_seeds),
    "hpo_budget_unit": "finite COMPLETE trials",
    "hpo_trials_per_repeat": "dimension-adaptive 3-5 COMPLETE trials",
    "stochastic_final_seeds": list(policy.lash_final_seeds),
    "historical_only_weather": True,
    "test_used_for_selection": False,
    "full_origin_final_refit_and_test": True,
    "inputs": input_rows,
    "hardware": hardware,
}
CONTRACT_PATH.write_text(json.dumps(contract, indent=2, default=str), encoding="utf-8")

with pd.ExcelWriter(OUTPUT_DIR / "computational_environment_and_protocol.xlsx", engine="openpyxl") as writer:
    pd.DataFrame([hardware]).to_excel(writer, sheet_name="Hardware", index=False)
    protocol_manifest(policy, config.dataset_keys).to_excel(writer, sheet_name="Protocol_Budget", index=False)
    search_space_manifest().to_excel(writer, sheet_name="Search_Spaces", index=False)
    pd.DataFrame(input_rows).to_excel(writer, sheet_name="Input_Hashes", index=False)

print("Frozen protocol:", CONTRACT_PATH)


## 4. Run the four-dataset benchmark

In [ ]:
RESULTS = run_per_dataset_hpo_benchmark(config, policy)
print("Completed datasets:", list(RESULTS))


## 5. Inspect the compact outputs

In [ ]:
summary_path = OUTPUT_DIR / "01_all_datasets_local_hpo_summary.xlsx"
print("Local-HPO summary:", summary_path)
if summary_path.exists():
    xls = pd.ExcelFile(summary_path)
    for sheet in xls.sheet_names:
        print("\n", sheet)
        display(pd.read_excel(summary_path, sheet_name=sheet).head(20))

for key in config.dataset_keys:
    book = OUTPUT_DIR / "benchmark" / key / f"{key}_benchmark_results.xlsx"
    print(key, "->", book)
